# Model Quality Checks (JupyterLite/Pyodide hosted copy)

**Module 8.3, Lesson 3 hands-on practice.**

This notebook is adapted from the open-source [Evidently AI ML Observability Course](https://github.com/evidentlyai/ml_observability_course) (module 5), used under the Apache License 2.0.

This JupyterLite version keeps the original lesson flow, but changes the runtime mechanics so the notebook works in a browser-based Pyodide kernel. It uses a local bank-marketing-compatible dataset, trains the model fresh in-session, and tries to use Evidently first. If Evidently cannot run in the browser kernel, the notebook uses a transparent local compatibility report for the same two teaching moments: model output checks before labels are available, and labelled model quality checks once targets are available.

**Run the next two cells first, in order.** The first tries to install Evidently in the JupyterLite kernel using `piplite`. The second writes the feature engineering helper and confirms which package versions are available in the browser. In JupyterLite, NumPy, pandas and scikit-learn are controlled by the Pyodide kernel, so this notebook does not try to force the Colab-pinned versions into the browser environment.


In [ ]:
# SETUP CELL 1: try to install Evidently in the JupyterLite/Pyodide kernel
# This uses the JupyterLite/Pyodide package-install route rather than a shell command.
# If Evidently cannot be installed in this browser build, the notebook will continue
# with browser-safe local teaching reports defined in the import cell below.

try:
    import evidently
    print(f"Evidently already available: {getattr(evidently, '__version__', 'version not reported')}")
except Exception:
    try:
        import piplite
        await piplite.install("evidently==0.4.19")
        import evidently
        print(f"Evidently installed: {getattr(evidently, '__version__', 'version not reported')}")
    except Exception as exc:
        print("Evidently could not be installed in this JupyterLite/Pyodide session.")
        print("The notebook will use browser-safe local teaching reports for the two TestSuite cells.")
        print("These reports keep the same decision points: output distribution, score drift, classification metrics and confusion matrices.")
        print(f"Reason: {type(exc).__name__}: {exc}")


In [ ]:
# SETUP CELL 2: write the feature engineering helper and verify available versions
helper = """import numpy as np
import pandas as pd

def feature_engineering(raw_data: pd.DataFrame) -> pd.DataFrame:
    preprocessed_data = raw_data.copy(deep = True)
    preprocessed_data.columns = ['age', 'job', 'marital', 'education', 'default', 'balance', 'housing', 'loan', 'contact', 
                               'day', 'month', 'duration', 'campaign', 'pdays', 'previous', 'poutcome', 'class']

    #client data preprocessing
    preprocessed_data['has_default'] = preprocessed_data.default.apply(
        lambda x : 0 if x == 'no' else 1 if x == 'yes' else -1)

    preprocessed_data['has_housing_loan'] = preprocessed_data.housing.apply(
        lambda x : 0 if x == 'no' else 1 if x == 'yes' else -1)

    preprocessed_data['has_pers_loan'] = preprocessed_data.loan.apply(
        lambda x : 0 if x == 'no' else 1 if x == 'yes' else -1)

    marital_dummies = pd.get_dummies(preprocessed_data.marital, prefix = 'marital')
    preprocessed_data = pd.concat([preprocessed_data, marital_dummies], axis=1)

    job_dummies = pd.get_dummies(preprocessed_data.job, prefix = 'job')
    preprocessed_data = pd.concat([preprocessed_data, job_dummies], axis=1)

    edu_dummies = pd.get_dummies(preprocessed_data.education, prefix = 'edu')
    preprocessed_data = pd.concat([preprocessed_data, edu_dummies], axis=1)

    preprocessed_data.drop(columns = ['default', 'housing','loan','marital', 'job','education'], inplace = True)

    #last contact data preprocessing
    contact_dummies = pd.get_dummies(preprocessed_data.contact, prefix = 'contact_type')
    preprocessed_data = pd.concat([preprocessed_data, contact_dummies], axis=1)

    lst_contact_mnth_dummies = pd.get_dummies(preprocessed_data.month, prefix = 'lst_contact_mnth')
    preprocessed_data = pd.concat([preprocessed_data, lst_contact_mnth_dummies], axis=1)

    preprocessed_data.drop(columns = ['contact', 'month'], inplace = True)

    #other attributes preprocessing
    prev_camp_outcome_dummies = pd.get_dummies(preprocessed_data.poutcome, prefix = 'prev_camp_outcome')
    preprocessed_data = pd.concat([preprocessed_data, prev_camp_outcome_dummies], axis=1)

    preprocessed_data.drop(columns = ['poutcome'], inplace = True)

    #target preprocessing
    preprocessed_data['target'] = preprocessed_data['class'].apply(lambda x : 0 if x == '1' else 1)
    preprocessed_data.drop(columns = ['class'], inplace = True)

    return preprocessed_data


def make_bank_marketing_like_data(n_rows: int = 13000, random_state: int = 42) -> pd.DataFrame:
    rng = np.random.default_rng(random_state)
    idx = np.arange(n_rows)
    shifted = idx >= 7000

    age = np.where(shifted, rng.normal(43, 12, n_rows), rng.normal(39, 11, n_rows)).clip(18, 80).round().astype(int)
    balance = np.where(shifted, rng.normal(900, 1500, n_rows), rng.normal(1350, 1800, n_rows)).round().astype(int)
    duration = np.where(shifted, rng.gamma(2.0, 85, n_rows), rng.gamma(2.2, 95, n_rows)).clip(10, 1300).round().astype(int)
    campaign = np.where(shifted, rng.poisson(3.1, n_rows) + 1, rng.poisson(2.2, n_rows) + 1)
    previous = np.where(shifted, rng.poisson(0.35, n_rows), rng.poisson(0.55, n_rows))
    pdays = np.where(previous > 0, rng.integers(1, 365, n_rows), -1)
    day = rng.integers(1, 31, n_rows)

    jobs = np.array(['admin.', 'blue-collar', 'technician', 'services', 'management', 'retired', 'student', 'self-employed'])
    job_probs_ref = np.array([0.14, 0.20, 0.18, 0.12, 0.18, 0.08, 0.05, 0.05])
    job_probs_cur = np.array([0.12, 0.25, 0.16, 0.13, 0.13, 0.09, 0.04, 0.08])

    job = np.where(shifted, rng.choice(jobs, n_rows, p=job_probs_cur), rng.choice(jobs, n_rows, p=job_probs_ref))
    marital = rng.choice(['single', 'married', 'divorced'], n_rows, p=[0.31, 0.56, 0.13])
    education = rng.choice(['primary', 'secondary', 'tertiary', 'unknown'], n_rows, p=[0.15, 0.52, 0.27, 0.06])
    default = rng.choice(['no', 'yes'], n_rows, p=[0.97, 0.03])
    housing = np.where(shifted, rng.choice(['no', 'yes'], n_rows, p=[0.48, 0.52]), rng.choice(['no', 'yes'], n_rows, p=[0.42, 0.58]))
    loan = rng.choice(['no', 'yes'], n_rows, p=[0.83, 0.17])
    contact = np.where(shifted, rng.choice(['cellular', 'telephone', 'unknown'], n_rows, p=[0.52, 0.18, 0.30]), rng.choice(['cellular', 'telephone', 'unknown'], n_rows, p=[0.67, 0.16, 0.17]))
    month = rng.choice(['jan', 'feb', 'mar', 'apr', 'may', 'jun', 'jul', 'aug', 'sep', 'oct', 'nov', 'dec'], n_rows, p=[0.06,0.05,0.04,0.08,0.14,0.11,0.12,0.11,0.07,0.07,0.09,0.06])
    poutcome = np.where(shifted, rng.choice(['success', 'failure', 'other', 'unknown'], n_rows, p=[0.08, 0.12, 0.08, 0.72]), rng.choice(['success', 'failure', 'other', 'unknown'], n_rows, p=[0.11, 0.14, 0.10, 0.65]))

    score = (
        -2.35
        + 0.004 * np.maximum(balance, -1000)
        + 0.008 * duration
        - 0.18 * campaign
        + 0.45 * (previous > 0)
        + 0.75 * (poutcome == 'success')
        + 0.35 * np.isin(job, ['student', 'retired', 'management'])
        - 0.35 * (housing == 'yes')
        - 0.45 * (loan == 'yes')
        - 0.55 * (default == 'yes')
        - 0.30 * shifted
    )
    probability = 1 / (1 + np.exp(-score))
    target = rng.binomial(1, probability)
    class_value = np.where(target == 0, '1', '2')

    return pd.DataFrame({
        'age': age,
        'job': job,
        'marital': marital,
        'education': education,
        'default': default,
        'balance': balance,
        'housing': housing,
        'loan': loan,
        'contact': contact,
        'day': day,
        'month': month,
        'duration': duration,
        'campaign': campaign,
        'pdays': pdays,
        'previous': previous,
        'poutcome': poutcome,
        'class': class_value,
    })
"""
with open("utils.py", "w") as f:
    f.write(helper)

import importlib, sys
packages = ["numpy", "pandas", "sklearn", "evidently"]
print(f"Python {sys.version.split()[0]}")
for pkg in packages:
    try:
        module = importlib.import_module(pkg)
        print(f"OK  {pkg}: {getattr(module, '__version__', 'version not reported')}")
    except Exception:
        if pkg == "evidently":
            print("INFO  evidently: not available; compatibility checks will be used")
        else:
            raise

print("\nEnvironment checked. Continue with the notebook.")


# Model Quality Checks

In [ ]:
import pandas as pd
import numpy as np
import pickle
from IPython.display import Markdown, display

from sklearn import ensemble
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    roc_auc_score,
    log_loss,
)

from utils import feature_engineering, make_bank_marketing_like_data

try:
    from evidently.test_suite import TestSuite
    from evidently.test_preset import NoTargetPerformanceTestPreset, BinaryClassificationTestPreset, DataDriftTestPreset
    USING_EVIDENTLY = True
    print("Using Evidently TestSuite.")
except Exception as evidently_error:
    USING_EVIDENTLY = False
    print("Evidently is not available in this browser kernel.")
    print("Using browser-safe local teaching reports for the same learning goals.")
    print(f"Evidently import issue: {type(evidently_error).__name__}: {evidently_error}")

    class NoTargetPerformanceTestPreset:
        name = "NoTargetPerformanceTestPreset"

    class BinaryClassificationTestPreset:
        name = "BinaryClassificationTestPreset"

    class DataDriftTestPreset:
        name = "DataDriftTestPreset"

    class TestSuite:
        """Small browser-safe teaching layer for this notebook.

        It keeps the original Evidently-style API used below:
        TestSuite(...).run(...).show(mode='inline')

        It does not attempt to reproduce the full Evidently UI. It produces focused
        reports that preserve the two core concepts in this lesson:
        1. No-target checks inspect model outputs before current labels are available.
        2. Target-aware checks inspect model quality once current labels are available.
        """

        def __init__(self, tests):
            self.tests = tests
            self.sections = []
            self.result = None
            self.suite_label = "compatibility report"

        def run(self, reference_data, current_data, column_mapping=None):
            self.sections = []
            frames = []

            for test in self.tests:
                if isinstance(test, NoTargetPerformanceTestPreset):
                    self.suite_label = "model output checks"
                    report = self._no_target_performance(reference_data, current_data)
                elif isinstance(test, BinaryClassificationTestPreset):
                    self.suite_label = "model quality checks"
                    report = self._binary_classification(reference_data, current_data)
                elif isinstance(test, DataDriftTestPreset):
                    self.suite_label = "data drift checks"
                    report = self._data_drift(reference_data, current_data)
                else:
                    report = {
                        "summary": pd.DataFrame([{
                            "check": type(test).__name__,
                            "status": "SKIPPED",
                            "detail": "No compatibility implementation for this test type.",
                        }])
                    }

                self.sections.extend(report.items())
                frames.extend(report.values())

            self.result = pd.concat(frames, ignore_index=True, sort=False) if frames else pd.DataFrame()
            return self

        def show(self, mode="inline"):
            if self.result is None:
                raise RuntimeError("Run the test suite before calling show().")

            display(Markdown(
                f"### Local teaching report: {self.suite_label}\n\n"
                "Evidently could not run in this browser kernel, so this cell is using a browser-safe local teaching report. "
                "It keeps the same learning question for this part of the notebook and shows the checks learners need to interpret."
            ))

            for title, table in self.sections:
                display(Markdown(f"**{title}**"))
                display(table)

            return None

        def _status_from_threshold(self, value, watch, investigate, higher_is_worse=True):
            if pd.isna(value):
                return "INFO"
            score = value if higher_is_worse else -value
            if score >= investigate:
                return "INVESTIGATE"
            if score >= watch:
                return "WATCH"
            return "OK"

        def _rates(self, series):
            values = series.value_counts(normalize=True).to_dict()
            return {0: values.get(0, 0.0), 1: values.get(1, 0.0)}

        def _safe_ks(self, reference_values, current_values):
            ref = pd.Series(reference_values).dropna().astype(float).to_numpy()
            cur = pd.Series(current_values).dropna().astype(float).to_numpy()
            if len(ref) == 0 or len(cur) == 0:
                return np.nan
            grid = np.sort(np.unique(np.concatenate([ref, cur])))
            ref_sorted = np.sort(ref)
            cur_sorted = np.sort(cur)
            ref_cdf = np.searchsorted(ref_sorted, grid, side="right") / len(ref_sorted)
            cur_cdf = np.searchsorted(cur_sorted, grid, side="right") / len(cur_sorted)
            return float(np.max(np.abs(ref_cdf - cur_cdf)))

        def _safe_psi(self, reference_values, current_values, bins=10):
            ref = pd.Series(reference_values).dropna().astype(float)
            cur = pd.Series(current_values).dropna().astype(float)
            if len(ref) == 0 or len(cur) == 0:
                return np.nan

            edges = np.unique(np.quantile(ref, np.linspace(0, 1, bins + 1)))
            if len(edges) < 3:
                lo = min(ref.min(), cur.min())
                hi = max(ref.max(), cur.max())
                if lo == hi:
                    return 0.0
                edges = np.linspace(lo, hi, min(bins, 3) + 1)

            edges[0] = -np.inf
            edges[-1] = np.inf
            ref_counts, _ = np.histogram(ref, bins=edges)
            cur_counts, _ = np.histogram(cur, bins=edges)
            eps = 1e-6
            ref_pct = ref_counts / max(ref_counts.sum(), 1) + eps
            cur_pct = cur_counts / max(cur_counts.sum(), 1) + eps
            return float(np.sum((cur_pct - ref_pct) * np.log(cur_pct / ref_pct)))

        def _no_target_performance(self, reference_data, current_data):
            concept = pd.DataFrame([
                {
                    "concept": "No-target model output check",
                    "what this can tell you": "Whether the model's predictions have changed compared with a reference period.",
                    "what this cannot tell you": "Whether the model is now right or wrong. That needs current target labels.",
                }
            ])

            ref_pred_rates = self._rates(reference_data["prediction"])
            cur_pred_rates = self._rates(current_data["prediction"])
            positive_delta = cur_pred_rates[1] - ref_pred_rates[1]

            checks = [
                {
                    "check": "row count",
                    "reference": len(reference_data),
                    "current": len(current_data),
                    "change": len(current_data) - len(reference_data),
                    "status": "INFO",
                    "interpretation": "Confirms the batch sizes being compared.",
                },
                {
                    "check": "positive prediction rate",
                    "reference": ref_pred_rates[1],
                    "current": cur_pred_rates[1],
                    "change": positive_delta,
                    "status": self._status_from_threshold(abs(positive_delta), watch=0.05, investigate=0.10),
                    "interpretation": "Large movement may mean the incoming population, model behaviour or threshold assumptions have changed.",
                },
                {
                    "check": "negative prediction rate",
                    "reference": ref_pred_rates[0],
                    "current": cur_pred_rates[0],
                    "change": cur_pred_rates[0] - ref_pred_rates[0],
                    "status": "INFO",
                    "interpretation": "Complements the positive prediction rate for a binary classifier.",
                },
            ]

            if "prediction_proba_1" in reference_data.columns and "prediction_proba_1" in current_data.columns:
                ref_score = reference_data["prediction_proba_1"]
                cur_score = current_data["prediction_proba_1"]
                score_mean_delta = cur_score.mean() - ref_score.mean()
                score_ks = self._safe_ks(ref_score, cur_score)
                score_psi = self._safe_psi(ref_score, cur_score)

                checks.extend([
                    {
                        "check": "mean positive-class score",
                        "reference": ref_score.mean(),
                        "current": cur_score.mean(),
                        "change": score_mean_delta,
                        "status": self._status_from_threshold(abs(score_mean_delta), watch=0.03, investigate=0.07),
                        "interpretation": "The score can move even when the hard class label changes only slightly.",
                    },
                    {
                        "check": "score distribution KS distance",
                        "reference": 0.0,
                        "current": score_ks,
                        "change": score_ks,
                        "status": self._status_from_threshold(score_ks, watch=0.10, investigate=0.20),
                        "interpretation": "Compares the whole score distribution, not just the mean.",
                    },
                    {
                        "check": "score distribution PSI",
                        "reference": 0.0,
                        "current": score_psi,
                        "change": score_psi,
                        "status": self._status_from_threshold(score_psi, watch=0.10, investigate=0.20),
                        "interpretation": "Population Stability Index gives a compact drift-style signal for score movement.",
                    },
                ])

            if "prediction_confidence" in reference_data.columns and "prediction_confidence" in current_data.columns:
                ref_conf = reference_data["prediction_confidence"]
                cur_conf = current_data["prediction_confidence"]
                low_ref = (ref_conf < 0.60).mean()
                low_cur = (cur_conf < 0.60).mean()
                checks.extend([
                    {
                        "check": "mean prediction confidence",
                        "reference": ref_conf.mean(),
                        "current": cur_conf.mean(),
                        "change": cur_conf.mean() - ref_conf.mean(),
                        "status": self._status_from_threshold(ref_conf.mean() - cur_conf.mean(), watch=0.03, investigate=0.07),
                        "interpretation": "A fall in confidence can be an early warning before labels arrive.",
                    },
                    {
                        "check": "low-confidence prediction rate",
                        "reference": low_ref,
                        "current": low_cur,
                        "change": low_cur - low_ref,
                        "status": self._status_from_threshold(low_cur - low_ref, watch=0.05, investigate=0.10),
                        "interpretation": "More low-confidence predictions may indicate the model is seeing less familiar cases.",
                    },
                ])

            details = pd.DataFrame(checks)
            for column in ["reference", "current", "change"]:
                details[column] = pd.to_numeric(details[column], errors="coerce")
            return {
                "Concept being checked": concept,
                "Output checks": details,
            }

        def _classification_metrics(self, frame):
            y_true = frame["target"].astype(int)
            y_pred = frame["prediction"].astype(int)
            tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()

            metrics = {
                "target positive rate": y_true.mean(),
                "prediction positive rate": y_pred.mean(),
                "accuracy": accuracy_score(y_true, y_pred),
                "precision": precision_score(y_true, y_pred, zero_division=0),
                "recall": recall_score(y_true, y_pred, zero_division=0),
                "f1": f1_score(y_true, y_pred, zero_division=0),
                "false positive rate": fp / max(fp + tn, 1),
                "false negative rate": fn / max(fn + tp, 1),
            }

            if "prediction_proba_1" in frame.columns:
                proba = frame["prediction_proba_1"].clip(1e-6, 1 - 1e-6)
                try:
                    metrics["roc auc"] = roc_auc_score(y_true, proba)
                except ValueError:
                    metrics["roc auc"] = np.nan
                metrics["log loss"] = log_loss(y_true, proba, labels=[0, 1])

            confusion = pd.DataFrame([
                {"actual": 0, "predicted_0": tn, "predicted_1": fp, "meaning": "true negatives / false positives"},
                {"actual": 1, "predicted_0": fn, "predicted_1": tp, "meaning": "false negatives / true positives"},
            ])
            return metrics, confusion

        def _binary_classification(self, reference_data, current_data):
            concept = pd.DataFrame([
                {
                    "concept": "Target-aware model quality check",
                    "what this can tell you": "Whether predictions are still correct on labelled current data.",
                    "why it follows the output check": "Output drift is only a warning. Labelled quality checks show whether the warning affected actual performance.",
                }
            ])

            ref_metrics, ref_confusion = self._classification_metrics(reference_data)
            cur_metrics, cur_confusion = self._classification_metrics(current_data)
            rows = []

            lower_is_worse = {"accuracy", "precision", "recall", "f1", "roc auc"}
            higher_is_worse = {"false positive rate", "false negative rate", "log loss"}

            for metric in ref_metrics:
                ref_value = ref_metrics[metric]
                cur_value = cur_metrics.get(metric, np.nan)
                change = cur_value - ref_value

                if metric in lower_is_worse:
                    severity = ref_value - cur_value
                    status = self._status_from_threshold(severity, watch=0.03, investigate=0.07)
                    interpretation = "Performance metric. A material drop should be investigated."
                elif metric in higher_is_worse:
                    severity = cur_value - ref_value
                    status = self._status_from_threshold(severity, watch=0.03, investigate=0.07)
                    interpretation = "Error metric. A material rise should be investigated."
                else:
                    severity = abs(change)
                    status = self._status_from_threshold(severity, watch=0.05, investigate=0.10)
                    interpretation = "Rate comparison. Movement changes how the model behaves for this batch."

                rows.append({
                    "metric": metric,
                    "reference": ref_value,
                    "current": cur_value,
                    "change": change,
                    "status": status,
                    "interpretation": interpretation,
                })

            metrics = pd.DataFrame(rows)
            confusion = pd.concat([
                ref_confusion.assign(dataset="reference"),
                cur_confusion.assign(dataset="current"),
            ], ignore_index=True)
            confusion = confusion[["dataset", "actual", "predicted_0", "predicted_1", "meaning"]]

            return {
                "Concept being checked": concept,
                "Metric comparison": metrics,
                "Confusion matrices": confusion,
            }

        def _data_drift(self, reference_data, current_data):
            numeric_columns = reference_data.select_dtypes(include=[np.number]).columns
            rows = []
            for column in numeric_columns:
                if column in ["target", "prediction", "prediction_proba_0", "prediction_proba_1", "prediction_confidence"]:
                    continue
                ref = reference_data[column].astype(float)
                cur = current_data[column].astype(float)
                pooled_std = np.sqrt((ref.var(ddof=0) + cur.var(ddof=0)) / 2)
                smd = 0.0 if pooled_std == 0 else (cur.mean() - ref.mean()) / pooled_std
                psi = self._safe_psi(ref, cur)
                rows.append({
                    "feature": column,
                    "reference_mean": ref.mean(),
                    "current_mean": cur.mean(),
                    "standardised_mean_difference": smd,
                    "psi": psi,
                    "status": "INVESTIGATE" if abs(smd) >= 0.30 or psi >= 0.20 else "WATCH" if abs(smd) >= 0.15 or psi >= 0.10 else "OK",
                })
            drift = pd.DataFrame(rows).sort_values(["status", "standardised_mean_difference"], ascending=[True, False])
            return {"Feature drift summary": drift.head(15)}


## Load Data

In [ ]:
# JupyterLite/Pyodide version: use a fixed local dataset instead of a runtime OpenML request.
# This keeps the notebook self-contained and avoids a runtime network dependency.
bank_marketing_data = make_bank_marketing_like_data(n_rows=13000, random_state=42)
bank_marketing_data.head()


In [ ]:
processed_data = feature_engineering(bank_marketing_data)

In [ ]:
processed_data_train = processed_data[:5000].copy()
processed_data_reference = processed_data[5000:7000].copy()
processed_data_prod_simulation = processed_data[7000:].copy()
batch_size = 2000


In [ ]:
processed_data_train.shape

## Train the model

The original course loads a previously saved model from a file. Here you train it fresh instead. This takes a few seconds and guarantees the model matches the browser kernel's available library versions.


In [ ]:
import pickle
feature_columns = processed_data_train.drop(columns=['target']).columns.tolist()
model = ensemble.RandomForestClassifier(random_state=42, n_estimators=100, n_jobs=1)
model.fit(processed_data_train[feature_columns], processed_data_train['target'])
with open("model2.pckl", "wb") as fout:
    pickle.dump(model, fout)
print("Model trained and saved.")

In [ ]:
processed_data_reference['prediction'] = model.predict(processed_data_reference[feature_columns])
processed_data_prod_simulation['prediction'] = model.predict(processed_data_prod_simulation[feature_columns])

# Evidently can use prediction scores as well as hard labels. The local JupyterLite checks use
# these columns to teach score-distribution monitoring, which is the key idea behind no-target output checks.
reference_proba = model.predict_proba(processed_data_reference[feature_columns])
current_proba = model.predict_proba(processed_data_prod_simulation[feature_columns])
processed_data_reference['prediction_proba_0'] = reference_proba[:, 0]
processed_data_reference['prediction_proba_1'] = reference_proba[:, 1]
processed_data_reference['prediction_confidence'] = reference_proba.max(axis=1)
processed_data_prod_simulation['prediction_proba_0'] = current_proba[:, 0]
processed_data_prod_simulation['prediction_proba_1'] = current_proba[:, 1]
processed_data_prod_simulation['prediction_confidence'] = current_proba.max(axis=1)

## Model Output Checks

In [ ]:
no_target_performance_suite = TestSuite(tests=[NoTargetPerformanceTestPreset()])
no_target_performance_suite.run(reference_data=processed_data_reference, current_data=processed_data_prod_simulation[2*batch_size:3*batch_size])
no_target_performance_suite.show(mode='inline')

## Model Quality Checks

In [ ]:
model_performance_suite = TestSuite(tests=[BinaryClassificationTestPreset()])
model_performance_suite.run(reference_data=processed_data_reference, current_data=processed_data_prod_simulation[0*batch_size:1*batch_size])
model_performance_suite.show(mode='inline')